
# Classical DE + Enrichment vs SDAN (Su_2020)

This notebook implements the mandatory comparison requested in reviewer comment:

- Differential expression on **training cells only**
- Enrichment on **GO BP / Reactome / immune terms**
- Redundancy collapse via **Jaccard-based filtering**
- Top-`K` non-redundant term selection
- Per-cell pathway activity scoring with **AddModuleScore-like scoring** (`scanpy.tl.score_genes`)
- Logistic regression on pathway scores
- Donor-level aggregation by averaging cell-level predictions

It reports:

- Cell-level and donor-level AUC
- Number of enriched terms
- Redundancy metric (median pairwise Jaccard among selected terms)
- An overlap example where DE+enrichment yields many overlapping terms, contrasted with SDAN modules


In [1]:

# If needed:
# !pip install scanpy statsmodels scikit-learn scipy matplotlib seaborn

import os
import csv
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from SDAN.preprocess import construct_gene_list

warnings.filterwarnings("ignore")
np.random.seed(888)
sc.settings.verbosity = 1


/Users/zxlin/Documents/GitHub/SDAN/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ------------------------
# Configuration
# ------------------------
ROOT = Path(".").resolve()
DATASET = "cd4_BL"      # "cd4_BL" or "cd8_BL"
GRAPH_WEIGHT = "2.0"    # SDAN output to compare against

DE_PADJ_CUTOFF = 0.05
DE_LOGFC_MIN = 0.0
MIN_OVERLAP = 5
JACCARD_THRESHOLD = 0.5
TOP_K = 40

data_dir = ROOT / "Su_2020"
anno_dir = ROOT / "Annotation"
out_dir = data_dir / "output"
fig_dir = data_dir / "figures"
fig_dir.mkdir(exist_ok=True)
out_dir.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATASET:", DATASET)


ROOT: /Users/zxlin/Documents/GitHub/SDAN
DATASET: cd4_BL


In [3]:
# ------------------------
# Helpers
# ------------------------

def parse_gmt(path):
    terms = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            term = parts[0]
            genes = set([g for g in parts[2:] if g])
            if genes:
                terms[term] = genes
    return terms


def jaccard(a, b):
    union = len(a | b)
    if union == 0:
        return 0.0
    return len(a & b) / union


def median_pairwise_jaccard(term_genes):
    vals = []
    n = len(term_genes)
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(jaccard(term_genes[i], term_genes[j]))
    return float(np.median(vals)) if vals else 0.0


def connected_components(edges, nodes):
    # edges: list[(a,b)] over node indices
    graph = {i: set() for i in nodes}
    for a, b in edges:
        graph[a].add(b)
        graph[b].add(a)
    seen = set()
    comps = []
    for n in nodes:
        if n in seen:
            continue
        stack = [n]
        comp = []
        seen.add(n)
        while stack:
            cur = stack.pop()
            comp.append(cur)
            for nxt in graph[cur]:
                if nxt not in seen:
                    seen.add(nxt)
                    stack.append(nxt)
        comps.append(sorted(comp))
    return sorted(comps, key=len, reverse=True)

In [4]:

# ------------------------
# Load Su_2020 data and construct labels/splits
# ------------------------

mtx_path = data_dir / f"gex_{DATASET}.mtx.gz"
gene_path = data_dir / f"gex_{DATASET}_genes.txt"
cell_path = data_dir / f"cell_info_{DATASET}.csv"
meta_path = data_dir / "Table_S1.xlsx"

adata = sc.read(mtx_path)
meta_cell = pd.read_csv(cell_path)
gene_names = pd.read_csv(gene_path, header=None).squeeze().astype(str)
meta_ind = pd.read_excel(meta_path, sheet_name="S1.1 Patient Clinical Data")

adata.var_names = gene_names.values
adata.obs_names = meta_cell["V1"].values
adata.obs["barcode"] = meta_cell["V1"].values
adata.obs["individual"] = meta_cell["individual"].values

# Same preprocessing as Su_2020.py
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

mito = pd.read_csv(anno_dir / "mito_genes.tsv", sep="	")
adata = adata[:, ~adata.var_names.isin(mito["hgnc_symbol"])].copy()

nonzero_prop = np.asarray((adata.X != 0).sum(axis=0)).ravel() / adata.shape[0]
adata = adata[:, nonzero_prop > 0.02].copy()

# Build mild/severe labels
meta_ind = meta_ind.copy()
meta_ind["Who Ordinal Scale"] = meta_ind["Who Ordinal Scale"].replace("1 or 2", 2)
meta_wos = meta_ind.groupby("Study Subject ID")["Who Ordinal Scale"].max()

mild_ind = meta_wos[meta_wos <= 2].index.to_series()
severe_ind = meta_wos[meta_wos >= 5].index.to_series()

adata.obs["cell_type"] = np.select(
    [
        adata.obs["individual"].isin(mild_ind),
        adata.obs["individual"].isin(severe_ind),
    ],
    ["mild", "severe"],
    default="moderate",
)

adata = adata[adata.obs["cell_type"].isin(["mild", "severe"])].copy()

# Match split logic in Su_2020.py (seeded)
test_ind = pd.concat([
    mild_ind.sample(n=math.floor(0.5 * len(mild_ind))),
    severe_ind.sample(n=math.floor(0.5 * len(severe_ind))),
])
train_ind = pd.concat([mild_ind, severe_ind]).drop(test_ind.index)

train_mask = adata.obs["individual"].isin(train_ind)
test_mask = adata.obs["individual"].isin(test_ind)

train_adata = adata[train_mask].copy()
test_adata = adata[test_mask].copy()

train_adata.obs["cell_type"] = train_adata.obs["cell_type"].astype("category")
test_adata.obs["cell_type"] = test_adata.obs["cell_type"].astype("category")

print("Full data shape:", adata.shape)
print("Train cells:", train_adata.n_obs, "Test cells:", test_adata.n_obs)
print("Train donors:", train_adata.obs['individual'].nunique(), "Test donors:", test_adata.obs['individual'].nunique())
print("Train label counts:\n", train_adata.obs['cell_type'].value_counts())


Full data shape: (42021, 7704)
Train cells: 19914 Test cells: 22107
Train donors: 39 Test donors: 38
Train label counts:
 cell_type
mild      12700
severe     7214
Name: count, dtype: int64


In [5]:
# ------------------------
# DE on training cells only (same method family as Su_2020.py via construct_gene_list)
# ------------------------

cell_type_list = ["mild", "severe"]
train_adata.obs["cell_type"] = train_adata.obs["cell_type"].astype("category")
train_adata.obs["cell_type"] = train_adata.obs["cell_type"].cat.set_categories(cell_type_list)

# Use SDAN preprocessing utility directly to define DE genes.
# n_top_genes is only used inside each class after FDR filtering.
n_top_genes = 1000
de_genes_idx = construct_gene_list(
    data=train_adata,
    cell_type_list=cell_type_list,
    n_top_genes=n_top_genes,
    method="fdr_bh",
    alpha=DE_PADJ_CUTOFF,
)
de_genes = set(de_genes_idx.astype(str))

# Build a compact DE table for downstream reporting convenience
de_df = pd.DataFrame({"gene": sorted(de_genes)})

print("DE gene list source: SDAN.preprocess.construct_gene_list")
print("n_top_genes per class (post-FDR):", n_top_genes)
print("Significant+selected DE genes (union):", len(de_genes))
de_df.head()

The number of DE genes for mild: 1184
The number of DE genes for severe: 2165
The number of DE genes: 2000
DE gene list source: SDAN.preprocess.construct_gene_list
n_top_genes per class (post-FDR): 1000
Significant+selected DE genes (union): 2000


,gene
0,ABCC1
1,ABCC10
2,ABCD3
3,ABCF1
4,ABCG1


In [6]:

# ------------------------
# Enrichment: GO BP + Reactome + immune terms
# ------------------------

gmt_files = {
    "GO_BP": anno_dir / "c5.go.bp.v2023.2.Hs.symbols.gmt",
    "Reactome": anno_dir / "c2.cp.reactome.v2023.2.Hs.symbols.gmt",
    "Immune": anno_dir / "c7.all.v2023.2.Hs.symbols.gmt",
}

terms = {}
for source, path in gmt_files.items():
    d = parse_gmt(path)
    for term, genes in d.items():
        terms[f"{source}::{term}"] = genes

universe = set(train_adata.var_names.astype(str))
N = len(de_genes & universe)
M = len(universe)

records = []
for term_name, genes in terms.items():
    gs = genes & universe
    n = len(gs)
    if n < MIN_OVERLAP:
        continue
    k = len(gs & de_genes)
    if k < MIN_OVERLAP:
        continue
    pval = hypergeom.sf(k - 1, M, n, N)
    records.append((term_name, n, k, pval, gs))

enrich = pd.DataFrame(records, columns=["term", "set_size", "overlap", "pval", "genes"])
if enrich.empty:
    raise RuntimeError("No enriched terms found. Try lowering MIN_OVERLAP or DE thresholds.")

enrich["padj"] = multipletests(enrich["pval"], method="fdr_bh")[1]
enrich = enrich.sort_values(["padj", "pval", "overlap"], ascending=[True, True, False]).reset_index(drop=True)

sig_enrich = enrich[enrich["padj"] < 0.05].copy()
print("Enriched terms (FDR < 0.05):", sig_enrich.shape[0])
sig_enrich.head(10)


Enriched terms (FDR < 0.05): 2072


,term,set_size,overlap,pval,genes,padj
0,Immune::ZAK_PBMC_MRKAD5_HIV_1_GAG_POL_NEF_AGE_...,690,305,8.841826e-28,"{PDE4DIP, AGMAT, KLHL28, NR1D2, CBLB, BBS2, LI...",7.417408e-24
1,Reactome::REACTOME_EUKARYOTIC_TRANSLATION_ELON...,89,67,1.823480e-22,"{RPL4, RPS18, RPS16, RPL32, RPS6, RPS12, RPL30...",7.648587e-19
2,Reactome::REACTOME_SELENOAMINO_ACID_METABOLISM,93,67,1.128805e-20,"{RPL4, RPS18, RPS16, RPL32, RPS6, RPS12, RPL30...",3.156515e-17
3,Immune::GSE14769_UNSTIM_VS_40MIN_LPS_BMDM_DN,112,75,5.952381e-20,"{PNRC1, UBC, IER3, CITED2, BIRC3, USP18, SPATS...",1.248363e-16
4,GO_BP::GOBP_CELL_ACTIVATION,519,225,3.930573e-19,"{ADA, HELLS, TBC1D10C, EIF2AK4, CD55, C12orf4,...",6.228008e-16
5,Reactome::REACTOME_RESPONSE_OF_EIF2AK4_GCN2_TO...,97,67,4.522787e-19,"{EIF2AK4, RPL4, RPS18, RPS16, RPL32, RPS6, RPS...",6.228008e-16
6,Immune::GSE14769_UNSTIM_VS_60MIN_LPS_BMDM_DN,117,76,5.196812e-19,"{PNRC1, IER3, CITED2, BIRC3, DHX15, CCL5, SIAH...",6.228008e-16
7,Reactome::REACTOME_EUKARYOTIC_TRANSLATION_INIT...,115,75,6.286084e-19,"{EIF3I, RPL4, RPS18, RPS16, RPL32, RPS6, RPS12...",6.591745e-16
8,GO_BP::GOBP_DEFENSE_RESPONSE,727,292,8.717508e-19,"{IRF9, ADA, MX2, DDX56, DHX16, DHX58, EIF2AK4,...",8.125686e-16
9,Immune::GSE26495_NAIVE_VS_PD1LOW_CD8_TCELL_DN,121,77,1.850593e-18,"{LAG3, SMAD3, CCL5, TTC38, BTD, GNLY, RASSF1, ...",1.552463e-15


In [7]:

# ------------------------
# Redundancy collapse (Jaccard) + top-K non-redundant terms
# ------------------------

selected_idx = []
for idx, row in sig_enrich.iterrows():
    g = row["genes"]
    keep = True
    for j in selected_idx:
        g2 = sig_enrich.loc[j, "genes"]
        if jaccard(g, g2) >= JACCARD_THRESHOLD:
            keep = False
            break
    if keep:
        selected_idx.append(idx)
    if len(selected_idx) >= TOP_K:
        break

selected = sig_enrich.loc[selected_idx].copy().reset_index(drop=True)
selected["n_genes_used"] = selected["genes"].apply(len)

selected_gene_sets = selected["genes"].tolist()
median_jacc = median_pairwise_jaccard(selected_gene_sets)

print("Total enriched terms:", sig_enrich.shape[0])
print("Selected non-redundant terms:", selected.shape[0])
print("Median pairwise Jaccard among selected terms:", round(median_jacc, 4))
selected[["term", "set_size", "overlap", "padj", "n_genes_used"]].head(15)


Total enriched terms: 2072
Selected non-redundant terms: 40
Median pairwise Jaccard among selected terms: 0.0333


,term,set_size,overlap,padj,n_genes_used
0,Immune::ZAK_PBMC_MRKAD5_HIV_1_GAG_POL_NEF_AGE_...,690,305,7.417408e-24,690
1,Reactome::REACTOME_EUKARYOTIC_TRANSLATION_ELON...,89,67,7.648587e-19,89
2,Immune::GSE14769_UNSTIM_VS_40MIN_LPS_BMDM_DN,112,75,1.248363e-16,112
3,GO_BP::GOBP_CELL_ACTIVATION,519,225,6.228008e-16,519
4,Immune::GSE14769_UNSTIM_VS_60MIN_LPS_BMDM_DN,117,76,6.228008e-16,117
5,GO_BP::GOBP_DEFENSE_RESPONSE,727,292,8.125686e-16,727
6,Immune::GSE26495_NAIVE_VS_PD1LOW_CD8_TCELL_DN,121,77,1.552463e-15,121
7,Immune::GSE24574_BCL6_HIGH_TFH_VS_TFH_CD4_TCEL...,171,97,4.812028e-15,171
8,Immune::GSE10325_LUPUS_CD4_TCELL_VS_LUPUS_MYEL...,184,101,2.077832e-14,184
9,Immune::GSE36476_CTRL_VS_TSST_ACT_72H_MEMORY_C...,165,93,3.567939e-14,165


In [8]:

# ------------------------
# Per-cell activity scores (AddModuleScore-like) + logistic regression
# ------------------------

selected = selected.copy()
selected["score_col"] = [f"term_score_{i:02d}" for i in range(selected.shape[0])]

for _, row in selected.iterrows():
    genes = sorted(list(row["genes"]))
    score_col = row["score_col"]
    # AddModuleScore-like program score (scanpy)
    sc.tl.score_genes(train_adata, gene_list=genes, score_name=score_col, random_state=0)
    sc.tl.score_genes(test_adata, gene_list=genes, score_name=score_col, random_state=0)

feature_cols = selected["score_col"].tolist()
X_train = train_adata.obs[feature_cols].to_numpy()
X_test = test_adata.obs[feature_cols].to_numpy()
y_train = (train_adata.obs["cell_type"].values == "severe").astype(int)
y_test = (test_adata.obs["cell_type"].values == "severe").astype(int)

clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)

test_cell_prob = clf.predict_proba(X_test)[:, 1]
cell_auc = roc_auc_score(y_test, test_cell_prob)

test_pred_df = pd.DataFrame({
    "individual": test_adata.obs["individual"].values,
    "label": y_test,
    "pred_prob": test_cell_prob,
})

donor_df = test_pred_df.groupby("individual", as_index=False).agg(
    donor_pred_prob=("pred_prob", "mean"),
    donor_label=("label", "max"),
    n_cells=("label", "size"),
)
donor_auc = roc_auc_score(donor_df["donor_label"], donor_df["donor_pred_prob"])

print(f"Cell-level AUC (classical):  {cell_auc:.4f}")
print(f"Donor-level AUC (classical): {donor_auc:.4f}")
donor_df.head()


Cell-level AUC (classical):  0.7619
Donor-level AUC (classical): 0.8899


,individual,donor_pred_prob,donor_label,n_cells
0,INCOV002,0.566876,1,4263
1,INCOV013,0.493319,1,562
2,INCOV026,0.436610,1,535
3,INCOV029,0.407719,1,689
4,INCOV033,0.654044,1,239


In [9]:

# ------------------------
# Compare against SDAN outputs (same dataset/split family)
# ------------------------

sdan_cell_file = out_dir / f"score_{DATASET}_{GRAPH_WEIGHT}.npy"
sdan_donor_file = out_dir / f"score_ind_{DATASET}_{GRAPH_WEIGHT}.npy"
sdan_module_file = out_dir / f"name_s_{DATASET}_{GRAPH_WEIGHT}.txt"

sdan_cell_auc = np.nan
sdan_donor_auc = np.nan
n_sdan_modules = np.nan

if sdan_cell_file.exists():
    arr = np.load(sdan_cell_file)
    # columns: prob_class0, prob_class1, label
    sdan_cell_auc = roc_auc_score(arr[:, 2], arr[:, 1])

if sdan_donor_file.exists():
    arr = np.load(sdan_donor_file)
    # columns: label, score
    sdan_donor_auc = roc_auc_score(arr[:, 0], arr[:, 1])

if sdan_module_file.exists():
    with open(sdan_module_file) as f:
        rows = list(csv.reader(f))
    n_sdan_modules = sum(any(x.strip() for x in r) for r in rows)

comparison = pd.DataFrame([
    {
        "method": "Classical DE + enrichment baseline",
        "cell_auc": cell_auc,
        "donor_auc": donor_auc,
        "n_selected_terms_or_modules": selected.shape[0],
        "redundancy_median_jaccard": median_jacc,
        "n_total_enriched_terms": int(sig_enrich.shape[0]),
    },
    {
        "method": f"SDAN (graph_weight={GRAPH_WEIGHT})",
        "cell_auc": sdan_cell_auc,
        "donor_auc": sdan_donor_auc,
        "n_selected_terms_or_modules": n_sdan_modules,
        "redundancy_median_jaccard": np.nan,
        "n_total_enriched_terms": np.nan,
    },
])

comparison


,method,cell_auc,donor_auc,n_selected_terms_or_modules,redundancy_median_jaccard,n_total_enriched_terms
0,Classical DE + enrichment baseline,0.761921,0.889881,40,0.033307,2072.0
1,SDAN (graph_weight=2.0),0.896689,0.949405,40,NaN,NaN


In [10]:

# ------------------------
# Save deliverables for manuscript tables/figures
# ------------------------

selected_export = selected[["term", "set_size", "overlap", "pval", "padj", "n_genes_used", "score_col"]].copy()
selected_export.to_csv(out_dir / f"selected_terms_{DATASET}.tsv", sep="	", index=False)

sig_export = sig_enrich[["term", "set_size", "overlap", "pval", "padj"]].copy()
sig_export.to_csv(out_dir / f"all_enriched_terms_{DATASET}.tsv", sep="	", index=False)

comparison.to_csv(out_dir / f"comparison_{DATASET}.tsv", sep="	", index=False)
donor_df.to_csv(out_dir / f"donor_predictions_{DATASET}.tsv", sep="	", index=False)

summary = pd.Series({
    "dataset": DATASET,
    "cell_auc_classical": cell_auc,
    "donor_auc_classical": donor_auc,
    "n_total_enriched_terms": int(sig_enrich.shape[0]),
    "n_selected_nonredundant_terms": int(selected.shape[0]),
    "median_pairwise_jaccard_selected": float(median_jacc),
    "cell_auc_sdan": float(sdan_cell_auc) if pd.notna(sdan_cell_auc) else np.nan,
    "donor_auc_sdan": float(sdan_donor_auc) if pd.notna(sdan_donor_auc) else np.nan,
    "n_modules_sdan": float(n_sdan_modules) if pd.notna(n_sdan_modules) else np.nan,
})
summary.to_csv(out_dir / f"summary_{DATASET}.tsv", sep="	", header=False)

print("Saved:")
print(out_dir / f"selected_terms_{DATASET}.tsv")
print(out_dir / f"all_enriched_terms_{DATASET}.tsv")
print(out_dir / f"comparison_{DATASET}.tsv")
print(out_dir / f"donor_predictions_{DATASET}.tsv")
print(out_dir / f"summary_{DATASET}.tsv")
summary


Saved:
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/selected_terms_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/all_enriched_terms_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/comparison_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/donor_predictions_cd4_BL.tsv
/Users/zxlin/Documents/GitHub/SDAN/Su_2020/output/summary_cd4_BL.tsv


dataset                               cd4_BL
cell_auc_classical                  0.761921
donor_auc_classical                 0.889881
n_total_enriched_terms                  2072
n_selected_nonredundant_terms             40
median_pairwise_jaccard_selected    0.033307
cell_auc_sdan                       0.896689
donor_auc_sdan                      0.949405
n_modules_sdan                          40.0
dtype: object


## Notes for manuscript insertion

- Replace `[insert datasets]` with `Su_2020` (and additional cohorts once you run this notebook with other datasets).
- Replace `[insert AUCs]` with the printed `cell_auc_classical` / `donor_auc_classical` values.
- For the overlap example, cite the largest overlap component table and contrast with SDAN module count.
- To run CD8 baseline, set `DATASET = "cd8_BL"` and re-run all cells.
